Воспроизведение трейдов

In [1]:
from __future__ import annotations

import json
import math
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from src.data_io import load_binance_spot_1d_csv
from src.experiment_config import BacktestConfig
from src.metrics import COST_SCENARIOS
from src.backtest_engine.run import run_on_test, extract_test_returns
from src.strategy_registry import build_registry

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\.venv\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [ ]:
PROJECT_ROOT = Path('.').resolve()
OUT_DIR = PROJECT_ROOT / 'results' / 'poses_true'
OUT_DIR.mkdir(parents=True, exist_ok=True)

RETURN_ABS_TOL = 1e-10
MAX_FOLD_ROWS_PER_SOURCE = None 

SOURCES = [
    {
        'source_id': 'exp_1d',
        'timeframe': '1d',
        'scenario': 'u12',
        'folds_path': PROJECT_ROOT / 'results' / 'runner_exp' / '1d' / 'folds_best.parquet',
        'returns_path': PROJECT_ROOT / 'results' / 'runner_exp' / '1d' / 'returns_oos.parquet',
        'config_path': PROJECT_ROOT / 'results' / 'runner_exp' / '1d' / 'config_notebook_1d.json',
    },
    {
        'source_id': 'exp_4h',
        'timeframe': '4h',
        'scenario': 'u12',
        'folds_path': PROJECT_ROOT / 'results' / 'runner_exp' / '4h' / 'folds_best.parquet',
        'returns_path': PROJECT_ROOT / 'results' / 'runner_exp' / '4h' / 'returns_oos.parquet',
        'config_path': PROJECT_ROOT / 'results' / 'runner_exp' / '4h' / 'config_notebook_4h.json',
    },
    {
        'source_id': 'onchain_1d',
        'timeframe': '1d',
        'scenario': 'u17_onchain',
        'folds_path': PROJECT_ROOT / 'results' / 'runner_onchain' / 'folds_best.parquet',
        'returns_path': PROJECT_ROOT / 'results' / 'runner_onchain' / 'returns_oos.parquet',
        'config_path': PROJECT_ROOT / 'results' / 'runner_onchain' / 'config_runner_onchain.json',
    },
]

for s in SOURCES:
    for k in ['folds_path', 'returns_path', 'config_path']:
        if not Path(s[k]).exists():
            raise FileNotFoundError(s[k])

print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUT_DIR      =', OUT_DIR)

PROJECT_ROOT = C:\Users\prosh\OneDrive\Рабочий стол\git\diploma
OUT_DIR      = C:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\poses_true


In [3]:
GROUP_KEY = ['timeframe', 'scenario', 'symbol', 'cost', 'strategy_id']
ROW_KEY = GROUP_KEY + ['fold_id']
SERIES_KEY = ['date', 'symbol', 'cost', 'strategy_id', 'timeframe', 'scenario']


def to_naive_ts(x: Any) -> pd.Timestamp:
    ts = pd.Timestamp(x)
    if ts.tzinfo is not None:
        ts = ts.tz_convert(None)
    return ts


def normalize_date_col(df: pd.DataFrame) -> pd.Series:
    if 'date' in df.columns:
        s = pd.to_datetime(df['date'], utc=True, errors='coerce')
    elif 'datetime_utc' in df.columns:
        s = pd.to_datetime(df['datetime_utc'], utc=True, errors='coerce')
    else:
        raise ValueError('Expected date or datetime_utc column')
    return s.dt.tz_convert(None)


def normalize_returns_ref(df: pd.DataFrame, *, timeframe: str, scenario: str, source_id: str) -> pd.DataFrame:
    out = df.copy()
    out['date'] = normalize_date_col(out)
    out['ret_ref'] = pd.to_numeric(out['ret'], errors='coerce').fillna(0.0)
    out = out[['date', 'ret_ref', 'symbol', 'cost', 'strategy_id']].copy()
    out['timeframe'] = timeframe
    out['scenario'] = scenario
    out['source_id'] = source_id
    out = out.dropna(subset=['date']).sort_values(['timeframe', 'scenario', 'symbol', 'cost', 'strategy_id', 'date']).reset_index(drop=True)
    return out


def normalize_folds(df: pd.DataFrame, *, timeframe: str, scenario: str, source_id: str) -> pd.DataFrame:
    out = df.copy()
    needed = [
        'symbol', 'cost', 'fold_id', 'strategy_id', 'params',
        'buffer_start', 'test_start', 'test_end',
        'oos_n_bars', 'oos_trades', 'oos_exposure_pct',
    ]
    miss = [c for c in needed if c not in out.columns]
    if miss:
        raise ValueError(f'Missing fold columns: {miss}')

    out = out[needed].copy()
    for c in ['buffer_start', 'test_start', 'test_end']:
        out[c] = pd.to_datetime(out[c], utc=True, errors='coerce').map(to_naive_ts)

    out['oos_n_bars'] = pd.to_numeric(out['oos_n_bars'], errors='coerce')
    out['oos_trades'] = pd.to_numeric(out['oos_trades'], errors='coerce')
    out['oos_exposure_pct'] = pd.to_numeric(out['oos_exposure_pct'], errors='coerce')
    out['timeframe'] = timeframe
    out['scenario'] = scenario
    out['source_id'] = source_id
    out = out.sort_values(['symbol', 'cost', 'strategy_id', 'fold_id']).reset_index(drop=True)
    return out


_INT_RE = re.compile(r'^[-+]?\d+$')


def _coerce_scalar(v: Any) -> Any:
    if isinstance(v, str):
        s = v.strip()
        if _INT_RE.match(s):
            try:
                return int(s)
            except Exception:
                pass
        try:
            f = float(s)
            if math.isfinite(f):
                return f
        except Exception:
            return v
        return v
    return v


def _coerce_param_obj(obj: Any, *, key: str | None = None) -> Any:
    if isinstance(obj, dict):
        return {k: _coerce_param_obj(v, key=k) for k, v in obj.items()}
    if isinstance(obj, list):
        vals = [_coerce_param_obj(v, key=key) for v in obj]
        if key == 'ma_pair' and len(vals) == 2:
            return (int(float(vals[0])), int(float(vals[1])))
        return vals
    if isinstance(obj, tuple):
        vals = tuple(_coerce_param_obj(v, key=key) for v in obj)
        if key == 'ma_pair' and len(vals) == 2:
            return (int(float(vals[0])), int(float(vals[1])))
        return vals
    if key == 'ma_pair' and obj is not None:
        try:
            return int(float(obj))
        except Exception:
            return obj
    return _coerce_scalar(obj)


def parse_params(raw: Any) -> dict[str, Any]:
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return {}
    if isinstance(raw, dict):
        return _coerce_param_obj(raw)
    if isinstance(raw, str):
        s = raw.strip()
        if not s:
            return {}
        try:
            parsed = json.loads(s)
            if isinstance(parsed, dict):
                return _coerce_param_obj(parsed)
            return {}
        except Exception:
            return {}
    return {}

In [4]:
source_context: dict[str, dict[str, Any]] = {}
returns_ref_parts: list[pd.DataFrame] = []
folds_parts: list[pd.DataFrame] = []

registry_map = {
    '1d': {s.strategy_id: s for s in build_registry(timeframe='1d')},
    '4h': {s.strategy_id: s for s in build_registry(timeframe='4h')},
}

for src in SOURCES:
    cfg = json.loads(Path(src['config_path']).read_text(encoding='utf-8'))

    bt_cfg_raw = cfg.get('bt', {})
    bt_cfg = BacktestConfig(**bt_cfg_raw)

    data_paths_raw = cfg.get('data_paths', {})
    data_by_symbol = {}
    for sym, p_str in data_paths_raw.items():
        p = Path(p_str)
        data_by_symbol[sym] = load_binance_spot_1d_csv(p, project_root=PROJECT_ROOT)

    folds_df = normalize_folds(pd.read_parquet(src['folds_path']), timeframe=src['timeframe'], scenario=src['scenario'], source_id=src['source_id'])
    returns_ref_df = normalize_returns_ref(pd.read_parquet(src['returns_path']), timeframe=src['timeframe'], scenario=src['scenario'], source_id=src['source_id'])

    if MAX_FOLD_ROWS_PER_SOURCE is not None:
        folds_df = folds_df.head(int(MAX_FOLD_ROWS_PER_SOURCE)).copy()

    source_context[src['source_id']] = {
        'timeframe': src['timeframe'],
        'scenario': src['scenario'],
        'bt_cfg': bt_cfg,
        'data_by_symbol': data_by_symbol,
        'registry': registry_map[src['timeframe']],
    }

    folds_parts.append(folds_df)
    returns_ref_parts.append(returns_ref_df)

folds_all = pd.concat(folds_parts, ignore_index=True)
returns_ref_all = pd.concat(returns_ref_parts, ignore_index=True)

print('folds_all shape      =', folds_all.shape)
print('returns_ref_all shape=', returns_ref_all.shape)
print('sources loaded       =', list(source_context.keys()))
print('fold groups          =', folds_all.groupby(['timeframe', 'scenario'])['strategy_id'].nunique().to_dict())

folds_all shape      = (4026, 14)
returns_ref_all shape= (2301366, 8)
sources loaded       = ['exp_1d', 'exp_4h', 'onchain_1d']
fold groups          = {('1d', 'u12'): 12, ('1d', 'u17_onchain'): 5, ('4h', 'u12'): 12}


In [5]:
positions_fold_rows: list[pd.DataFrame] = []
equity_fold_rows: list[pd.DataFrame] = []
trades_fold_rows: list[pd.DataFrame] = []
meta_rows: list[dict[str, Any]] = []

n_total = len(folds_all)
print_every = 200

for i, row in enumerate(folds_all.itertuples(index=False), start=1):
    source_id = row.source_id
    tf = row.timeframe
    scenario = row.scenario
    sym = row.symbol
    cost_name = row.cost
    sid = row.strategy_id
    fold_id = int(row.fold_id)

    ctx = source_context[source_id]
    data_by_symbol = ctx['data_by_symbol']
    registry = ctx['registry']
    bt_cfg = ctx['bt_cfg']

    status = 'ok'
    err = None

    if i % print_every == 0 or i == 1 or i == n_total:
        print(f'Replay {i}/{n_total} ...')

    try:
        if sym not in data_by_symbol:
            raise KeyError(f'Missing symbol in data_paths: {sym}')
        if sid not in registry:
            raise KeyError(f'Missing strategy_id in registry({tf}): {sid}')
        if cost_name not in COST_SCENARIOS:
            raise KeyError(f'Missing cost scenario: {cost_name}')

        spec = registry[sid]
        params = parse_params(row.params)

        data = data_by_symbol[sym]
        buffer_start = to_naive_ts(row.buffer_start)
        test_start = to_naive_ts(row.test_start)
        test_end = to_naive_ts(row.test_end)

        test_df_with_buffer = data.loc[(data.index >= buffer_start) & (data.index < test_end)].copy()
        if test_df_with_buffer.empty:
            raise ValueError('test_df_with_buffer is empty')

        stats = run_on_test(
            test_df_with_buffer,
            strategy_cls=spec.cls,
            start_date=test_start,
            params=params,
            cost=COST_SCENARIOS[cost_name],
            bt_cfg=bt_cfg,
        )

        eq = stats['_equity_curve'].copy()
        if not isinstance(eq, pd.DataFrame):
            raise TypeError('_equity_curve is not DataFrame')

        eq_w = eq.reset_index().copy()
        dcol = 'date' if 'date' in eq_w.columns else ('datetime_utc' if 'datetime_utc' in eq_w.columns else eq_w.columns[0])
        eq_w['date'] = pd.to_datetime(eq_w[dcol], errors='coerce')
        eq_w['date'] = eq_w['date'].map(to_naive_ts)

        for c in ['Equity', 'DrawdownPct']:
            if c not in eq_w.columns:
                raise ValueError(f'Missing {c} in _equity_curve')

        n_eq = len(eq_w)
        pos_buf = np.zeros(n_eq, dtype=np.int8)

        trades = stats.get('_trades', None)
        if isinstance(trades, pd.DataFrame) and not trades.empty:
            tr = trades.copy()
            for tr_row in tr.itertuples(index=False):
                try:
                    en = int(getattr(tr_row, 'EntryBar'))
                    ex = int(getattr(tr_row, 'ExitBar'))
                except Exception:
                    continue
                en = max(0, min(en, n_eq - 1))
                ex = max(0, min(ex, n_eq - 1))
                if ex < en:
                    continue
                pos_buf[en:ex + 1] = 1
        else:
            tr = pd.DataFrame()

        eq_w['position'] = pos_buf

        ret_true = extract_test_returns(stats, test_start=test_start, test_end=test_end)
        ret_df = ret_true.rename('ret_true').reset_index().copy()
        rcol = 'date' if 'date' in ret_df.columns else ('datetime_utc' if 'datetime_utc' in ret_df.columns else ret_df.columns[0])
        ret_df['date'] = pd.to_datetime(ret_df[rcol], errors='coerce').map(to_naive_ts)
        ret_df = ret_df[['date', 'ret_true']]

        pure = eq_w.loc[(eq_w['date'] >= test_start) & (eq_w['date'] < test_end)].copy()
        pure = pure.merge(ret_df, on='date', how='left')
        pure['ret_true'] = pd.to_numeric(pure['ret_true'], errors='coerce').fillna(0.0)

        common_meta = {
            'source_id': source_id,
            'timeframe': tf,
            'scenario': scenario,
            'symbol': sym,
            'cost': cost_name,
            'strategy_id': sid,
            'fold_id': fold_id,
            'buffer_start': buffer_start,
            'test_start': test_start,
            'test_end': test_end,
        }

        pos_part = pure[['date', 'position', 'ret_true']].copy()
        for k, v in common_meta.items():
            pos_part[k] = v
        pos_part = pos_part[['date', 'position', 'ret_true', 'source_id', 'timeframe', 'scenario', 'symbol', 'cost', 'strategy_id', 'fold_id']]
        positions_fold_rows.append(pos_part)

        eq_part = pure[['date', 'Equity', 'DrawdownPct', 'DrawdownDuration', 'ret_true', 'position']].copy()
        eq_part = eq_part.rename(columns={'Equity': 'equity', 'DrawdownPct': 'drawdown_pct', 'DrawdownDuration': 'drawdown_duration'})
        for k, v in common_meta.items():
            eq_part[k] = v
        eq_part = eq_part[['date', 'equity', 'drawdown_pct', 'drawdown_duration', 'ret_true', 'position', 'source_id', 'timeframe', 'scenario', 'symbol', 'cost', 'strategy_id', 'fold_id']]
        equity_fold_rows.append(eq_part)

        n_trades_replay = 0
        if isinstance(tr, pd.DataFrame) and not tr.empty:
            tr_out = tr.copy()
            if 'EntryTime' in tr_out.columns:
                tr_out['EntryTime'] = pd.to_datetime(tr_out['EntryTime'], errors='coerce').map(to_naive_ts)
            if 'ExitTime' in tr_out.columns:
                tr_out['ExitTime'] = pd.to_datetime(tr_out['ExitTime'], errors='coerce').map(to_naive_ts)
            for k, v in common_meta.items():
                tr_out[k] = v
            trades_fold_rows.append(tr_out)
            n_trades_replay = int(len(tr_out))

        n_bars_test = int(len(pure))
        n_bars_ref = int(row.oos_n_bars) if pd.notna(row.oos_n_bars) else -1
        n_trades_ref = int(row.oos_trades) if pd.notna(row.oos_trades) else -1

        meta_rows.append({
            **common_meta,
            'params_raw': row.params,
            'params_norm': json.dumps(params, ensure_ascii=False, default=str),
            'n_bars_test_replay': n_bars_test,
            'n_bars_ref': n_bars_ref,
            'bars_match': bool(n_bars_test == n_bars_ref),
            'n_trades_replay': n_trades_replay,
            'n_trades_ref': n_trades_ref,
            'trades_match': bool(n_trades_replay == n_trades_ref),
            'status': status,
            'error': err,
        })

    except Exception as e:
        status = 'error'
        err = repr(e)
        meta_rows.append({
            'source_id': source_id,
            'timeframe': tf,
            'scenario': scenario,
            'symbol': sym,
            'cost': cost_name,
            'strategy_id': sid,
            'fold_id': fold_id,
            'buffer_start': to_naive_ts(row.buffer_start),
            'test_start': to_naive_ts(row.test_start),
            'test_end': to_naive_ts(row.test_end),
            'params_raw': row.params,
            'params_norm': None,
            'n_bars_test_replay': np.nan,
            'n_bars_ref': int(row.oos_n_bars) if pd.notna(row.oos_n_bars) else np.nan,
            'bars_match': False,
            'n_trades_replay': np.nan,
            'n_trades_ref': int(row.oos_trades) if pd.notna(row.oos_trades) else np.nan,
            'trades_match': False,
            'status': status,
            'error': err,
        })

meta_fold = pd.DataFrame(meta_rows)
positions_fold = pd.concat(positions_fold_rows, ignore_index=True) if positions_fold_rows else pd.DataFrame(columns=['date', 'position', 'ret_true', 'source_id', 'timeframe', 'scenario', 'symbol', 'cost', 'strategy_id', 'fold_id'])
equity_fold = pd.concat(equity_fold_rows, ignore_index=True) if equity_fold_rows else pd.DataFrame(columns=['date', 'equity', 'drawdown_pct', 'drawdown_duration', 'ret_true', 'position', 'source_id', 'timeframe', 'scenario', 'symbol', 'cost', 'strategy_id', 'fold_id'])
trades_fold = pd.concat(trades_fold_rows, ignore_index=True) if trades_fold_rows else pd.DataFrame()

print('positions_fold:', positions_fold.shape)
print('equity_fold   :', equity_fold.shape)
print('trades_fold   :', trades_fold.shape)
print('meta_fold     :', meta_fold.shape)
print('meta status   :', meta_fold['status'].value_counts(dropna=False).to_dict())

Replay 1/4026 ...
Replay 200/4026 ...
Replay 400/4026 ...
Replay 600/4026 ...
Replay 800/4026 ...
Replay 1000/4026 ...
Replay 1200/4026 ...
Replay 1400/4026 ...
Replay 1600/4026 ...
Replay 1800/4026 ...
Replay 2000/4026 ...
Replay 2200/4026 ...
Replay 2400/4026 ...
Replay 2600/4026 ...
Replay 2800/4026 ...
Replay 3000/4026 ...
Replay 3200/4026 ...
Replay 3400/4026 ...
Replay 3600/4026 ...
Replay 3800/4026 ...
Replay 4000/4026 ...
Replay 4026/4026 ...
positions_fold: (2301366, 10)
equity_fold   : (2301366, 13)
trades_fold   : (23403, 98)
meta_fold     : (4026, 20)
meta status   : {'ok': 4026}


In [ ]:
positions_fold.to_parquet(OUT_DIR / 'positions_true_fold.parquet', index=False)
equity_fold.to_parquet(OUT_DIR / 'equity_true_fold.parquet', index=False)
meta_fold.to_parquet(OUT_DIR / 'replay_meta_fold.parquet', index=False)
if trades_fold is not None and len(trades_fold):
    trades_fold.to_parquet(OUT_DIR / 'trades_true_fold.parquet', index=False)
else:
    pd.DataFrame(columns=['source_id', 'timeframe', 'scenario', 'symbol', 'cost', 'strategy_id', 'fold_id']).to_parquet(OUT_DIR / 'trades_true_fold.parquet', index=False)

positions_oos = positions_fold.drop(columns=['fold_id']).copy()
positions_oos = positions_oos.sort_values(['timeframe', 'scenario', 'symbol', 'cost', 'strategy_id', 'date'])

pre_dups = int(positions_oos.duplicated(SERIES_KEY).sum())
if pre_dups:
    print(f'Warning: duplicate rows before stitch dedupe: {pre_dups}')
positions_oos = positions_oos.drop_duplicates(SERIES_KEY, keep='first').reset_index(drop=True)
post_dups = int(positions_oos.duplicated(SERIES_KEY).sum())
assert post_dups == 0

equity_oos = equity_fold.drop(columns=['fold_id']).copy()
equity_oos = equity_oos.sort_values(['timeframe', 'scenario', 'symbol', 'cost', 'strategy_id', 'date'])
equity_oos = equity_oos.drop_duplicates(SERIES_KEY, keep='first').reset_index(drop=True)
assert int(equity_oos.duplicated(SERIES_KEY).sum()) == 0

positions_oos.to_parquet(OUT_DIR / 'positions_true_oos_long.parquet', index=False)
equity_oos.to_parquet(OUT_DIR / 'equity_true_oos_long.parquet', index=False)

print('positions_oos:', positions_oos.shape)
print('equity_oos   :', equity_oos.shape)

positions_oos: (2301366, 9)
equity_oos   : (2301366, 12)


In [ ]:
ret_true = positions_oos[['date', 'symbol', 'cost', 'strategy_id', 'timeframe', 'scenario', 'source_id', 'ret_true']].copy()
ret_ref = returns_ref_all[['date', 'symbol', 'cost', 'strategy_id', 'timeframe', 'scenario', 'source_id', 'ret_ref']].copy()

m = ret_ref.merge(
    ret_true,
    on=['date', 'symbol', 'cost', 'strategy_id', 'timeframe', 'scenario', 'source_id'],
    how='outer',
    indicator=True,
)

m['ret_ref'] = pd.to_numeric(m['ret_ref'], errors='coerce')
m['ret_true'] = pd.to_numeric(m['ret_true'], errors='coerce')
m['abs_diff'] = (m['ret_ref'] - m['ret_true']).abs()
m['is_common'] = m['_merge'].eq('both')
m['is_mismatch'] = m['is_common'] & (m['abs_diff'] > RETURN_ABS_TOL)

gcols = ['timeframe', 'scenario', 'symbol', 'cost', 'strategy_id', 'source_id']

ret_val = (
    m.groupby(gcols, dropna=False)
     .agg(
         n_ref=('ret_ref', lambda s: int(s.notna().sum())),
         n_true=('ret_true', lambda s: int(s.notna().sum())),
         n_common=('is_common', 'sum'),
         n_left_only=('_merge', lambda s: int((s == 'left_only').sum())),
         n_right_only=('_merge', lambda s: int((s == 'right_only').sum())),
         max_abs_diff=('abs_diff', lambda s: float(np.nanmax(s.to_numpy())) if s.notna().any() else np.nan),
         mae=('abs_diff', lambda s: float(np.nanmean(s.to_numpy())) if s.notna().any() else np.nan),
         n_mismatch=('is_mismatch', 'sum'),
     )
     .reset_index()
)

ret_val['pass_returns'] = (
    (ret_val['n_left_only'] == 0)
    & (ret_val['n_right_only'] == 0)
    & (ret_val['n_mismatch'] == 0)
    & (ret_val['max_abs_diff'] <= RETURN_ABS_TOL)
)

fold_val = meta_fold.copy()
fold_val['pass_bars'] = fold_val['bars_match'].fillna(False)
fold_val['pass_trades'] = fold_val['trades_match'].fillna(False)
fold_val['pass_status'] = fold_val['status'].eq('ok')
fold_val['pass_fold_all'] = fold_val['pass_status'] & fold_val['pass_bars'] & fold_val['pass_trades']

ret_fail = ret_val[~ret_val['pass_returns']].copy()
ret_fail['failure_type'] = 'returns'

fold_fail = fold_val[~fold_val['pass_fold_all']].copy()
fold_fail['failure_type'] = 'fold'

all_fail = pd.concat([ret_fail, fold_fail], ignore_index=True, sort=False)

ret_val.to_parquet(OUT_DIR / 'pos_true_validation_returns.parquet', index=False)
fold_val.to_parquet(OUT_DIR / 'pos_true_validation_folds.parquet', index=False)
all_fail.to_parquet(OUT_DIR / 'pos_true_validation_failures.parquet', index=False)

print('ret_val shape :', ret_val.shape)
print('fold_val shape:', fold_val.shape)
print('all_fail shape:', all_fail.shape)
print('ret pass rate :', float(ret_val['pass_returns'].mean()) if len(ret_val) else np.nan)
print('fold pass rate:', float(fold_val['pass_fold_all'].mean()) if len(fold_val) else np.nan)

ret_val shape : (420, 15)
fold_val shape: (4026, 24)
all_fail shape: (0, 34)
ret pass rate : 1.0
fold pass rate: 1.0


In [ ]:
coverage = (
    positions_oos.groupby(['timeframe', 'scenario'], as_index=False)
    .agg(
        n_symbols=('symbol', 'nunique'),
        n_strategies=('strategy_id', 'nunique'),
        n_costs=('cost', 'nunique'),
        n_rows=('position', 'size'),
    )
)

ret_pass_tf = (
    ret_val.groupby(['timeframe', 'scenario'], as_index=False)
    .agg(pass_returns_rate=('pass_returns', 'mean'))
)

fold_pass_tf = (
    fold_val.groupby(['timeframe', 'scenario'], as_index=False)
    .agg(
        pass_fold_all_rate=('pass_fold_all', 'mean'),
        pass_trades_rate=('pass_trades', 'mean'),
        pass_bars_rate=('pass_bars', 'mean'),
        pass_status_rate=('pass_status', 'mean'),
    )
)

print('Coverage')
display(coverage)

print('Return validation pass rates')
display(ret_pass_tf)

print('Fold validation pass rates')
display(fold_pass_tf)

print('Top return FAILs')
display(
    ret_val[~ret_val['pass_returns']]
    .sort_values(['max_abs_diff', 'n_mismatch'], ascending=[False, False])
    .head(30)
)

print('Top fold FAILs')
display(
    fold_val[~fold_val['pass_fold_all']]
    .sort_values(['status', 'symbol', 'cost', 'strategy_id', 'fold_id'])
    .head(30)
)

Coverage


,timeframe,scenario,n_symbols,n_strategies,n_costs,n_rows
0,1d,u12,5,12,3,315504
1,1d,u17_onchain,4,5,3,104070
2,4h,u12,5,12,3,1881792


Return validation pass rates


,timeframe,scenario,pass_returns_rate
0,1d,u12,1.0
1,1d,u17_onchain,1.0
2,4h,u12,1.0


Fold validation pass rates


,timeframe,scenario,pass_fold_all_rate,pass_trades_rate,pass_bars_rate,pass_status_rate
0,1d,u12,1.0,1.0,1.0,1.0
1,1d,u17_onchain,1.0,1.0,1.0,1.0
2,4h,u12,1.0,1.0,1.0,1.0


Top return FAILs


,timeframe,scenario,symbol,cost,strategy_id,source_id,n_ref,n_true,n_common,n_left_only,n_right_only,max_abs_diff,mae,n_mismatch,pass_returns


Top fold FAILs


,source_id,timeframe,scenario,symbol,cost,strategy_id,fold_id,buffer_start,test_start,test_end,params_raw,params_norm,n_bars_test_replay,n_bars_ref,bars_match,n_trades_replay,n_trades_ref,trades_match,status,error,pass_bars,pass_trades,pass_status,pass_fold_all


In [9]:
run_info = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'project_root': str(PROJECT_ROOT),
    'out_dir': str(OUT_DIR),
    'return_abs_tol': RETURN_ABS_TOL,
    'max_fold_rows_per_source': MAX_FOLD_ROWS_PER_SOURCE,
    'sources': [
        {
            'source_id': s['source_id'],
            'timeframe': s['timeframe'],
            'scenario': s['scenario'],
            'folds_path': str(s['folds_path']),
            'returns_path': str(s['returns_path']),
            'config_path': str(s['config_path']),
        }
        for s in SOURCES
    ],
    'shapes': {
        'folds_all': [int(folds_all.shape[0]), int(folds_all.shape[1])],
        'returns_ref_all': [int(returns_ref_all.shape[0]), int(returns_ref_all.shape[1])],
        'positions_true_fold': [int(positions_fold.shape[0]), int(positions_fold.shape[1])],
        'equity_true_fold': [int(equity_fold.shape[0]), int(equity_fold.shape[1])],
        'trades_true_fold': [int(trades_fold.shape[0]), int(trades_fold.shape[1])],
        'replay_meta_fold': [int(meta_fold.shape[0]), int(meta_fold.shape[1])],
        'positions_true_oos_long': [int(positions_oos.shape[0]), int(positions_oos.shape[1])],
        'equity_true_oos_long': [int(equity_oos.shape[0]), int(equity_oos.shape[1])],
        'pos_true_validation_returns': [int(ret_val.shape[0]), int(ret_val.shape[1])],
        'pos_true_validation_folds': [int(fold_val.shape[0]), int(fold_val.shape[1])],
        'pos_true_validation_failures': [int(all_fail.shape[0]), int(all_fail.shape[1])],
    },
}

(OUT_DIR / 'pos_true_run_info.json').write_text(
    json.dumps(run_info, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('Saved run info:', OUT_DIR / 'pos_true_run_info.json')

Saved run info: C:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\poses_true\pos_true_run_info.json
